In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

2026-01-11 09:23:40.559800: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-11 09:23:41.065013: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-11 09:23:42.932726: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/joe/miniconda3/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.p

In [ ]:
"""
train_regime_model.py
---------------------
Train ONE regime model with horizon as a feature.
Model type is pluggable: Gradient Boosting or Neural Net.

This script:
- Uses deseasonalized VPD anomalies
- Trains on all horizons (1–12)
- Includes horizon explicitly as an input
- Trains ONE model for ONE target-month regime
"""


# =========================
# CONFIG
# =========================
CSV_PATH = Path("full_data.csv")
DATE_COL = "Yrmo"          # YYYYMM
TARGET_COL = "VPD"        # change to "VPD" if needed

MAX_HORIZON = 12
MODEL_TYPE = "nn"          # "gb" or "nn"

REGIME_NAME = "july"
REGIME_MONTHS = {7}        # July-only

FEATURE_COLS = ["F1","F2","F3","F4","F5","F6"]        # set explicitly or auto-detect

RANDOM_STATE = 42


# =========================
# LOAD DATA
# =========================
df = pd.read_csv(CSV_PATH)

df["date"] = pd.to_datetime(df[DATE_COL].astype(str) + "01", format="%Y%m%d")
df["month"] = df["date"].dt.month
df = df.sort_values("date").reset_index(drop=True)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found")

# =========================
# DESEASONALIZE TARGET
# =========================
monthly_mean = df.groupby("month")[TARGET_COL].mean()
df["target_anom"] = df[TARGET_COL] - df["month"].map(monthly_mean)

# =========================
# FEATURE SELECTION
# =========================
if FEATURE_COLS is None:
    exclude = {
        DATE_COL, "date", "month",
        TARGET_COL, "target_anom"
    }
    FEATURE_COLS = [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]

if not FEATURE_COLS:
    raise RuntimeError("No numeric feature columns found")

print(f"Using {len(FEATURE_COLS)} features:")
print(FEATURE_COLS)

# =========================
# SCALE FEATURES & TARGET
# =========================
x_scaler = StandardScaler()
X_all = x_scaler.fit_transform(df[FEATURE_COLS].values.astype(np.float32))

y_scaler = StandardScaler()
y_all = y_scaler.fit_transform(
    df[["target_anom"]].values.astype(np.float32)
).ravel()

# =========================
# BUILD SUPERVISED PAIRS
# =========================
X_list = []
y_list = []
meta_month = []
meta_horizon = []
n = len(df)

for t in range(n):
    x_t = X_all[t]

    for h in range(1, MAX_HORIZON + 1):
        j = t + h
        if j >= n:
            continue

        target_month = int(df.loc[j, "month"])
        if target_month not in REGIME_MONTHS:
            continue

        # horizon as feature (scaled)
        h_feat = np.array([h / MAX_HORIZON], dtype=np.float32)

        X_list.append(np.concatenate([x_t, h_feat]))
        y_list.append(y_all[j])
        meta_month.append(target_month)
        meta_horizon.append(h)

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)
meta_month = np.array(meta_month)
meta_horizon = np.array(meta_horizon)

print(f"\nBuilt dataset for regime '{REGIME_NAME}':")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

if len(y) < 100:
    print("⚠️ WARNING: Very few training samples for this regime")

# =========================
# TRAIN / VALIDATION SPLIT
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)
split = int(0.8 * len(y))
meta_month_val = meta_month[split:]
meta_horizon_val = meta_horizon[split:]
# =========================
# MODEL FACTORY
# =========================

def quantile_loss(tau):
    """
    Quantile (pinball) loss.
    tau = 0.5 → MAE
    tau > 0.5 → penalize underprediction more
    """
    def loss(y_true, y_pred):
        e = y_true - y_pred
        return tf.reduce_mean(tf.maximum(tau * e, (tau - 1) * e))
    return loss

def build_model(model_type, input_dim):
    if model_type == "gb":
        from sklearn.ensemble import HistGradientBoostingRegressor
        return HistGradientBoostingRegressor(
            loss="absolute_error",
            max_depth=5,
            learning_rate=0.05,
            max_iter=400,
            random_state=RANDOM_STATE
        )

    elif model_type == "nn":
        

        TAU = 0.8   # <<< key control knob

        model = Sequential([
            Input(shape=(input_dim,)),
            Dense(32, activation="relu"),
            Dense(16, activation="relu"),
            Dense(1)
        ])

        model.compile(
            optimizer=Adam(1e-3),
            loss=quantile_loss(TAU)
        )



        return model

    else:
        raise ValueError(f"Unknown MODEL_TYPE '{model_type}'")

# =========================
# TRAIN MODEL
# =========================
model = build_model(MODEL_TYPE, X_train.shape[1])

if MODEL_TYPE == "gb":
    model.fit(X_train, y_train)

elif MODEL_TYPE == "nn":
    from tensorflow.keras.callbacks import EarlyStopping
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=300,
        batch_size=32,
        callbacks=[EarlyStopping(patience=20, restore_best_weights=True)],
        verbose=1
    )

# =========================
# SAVE ARTIFACTS
# =========================
out_prefix = f"{REGIME_NAME}_{MODEL_TYPE}"

if MODEL_TYPE == "gb":
    joblib.dump(model, f"model_{out_prefix}.pkl")
else:
    model.save(f"model_{out_prefix}.h5")

joblib.dump(x_scaler, f"x_scaler_{REGIME_NAME}.pkl")
joblib.dump(y_scaler, f"y_scaler_{REGIME_NAME}.pkl")
joblib.dump(monthly_mean, "monthly_mean.pkl")

print("\n✅ Training complete")
print(f"Saved model_{out_prefix}")


Using 6 features:
['F1', 'F2', 'F3', 'F4', 'F5', 'F6']

Built dataset for regime 'july':
X shape: (414, 7)
y shape: (414,)


I0000 00:00:1768148647.064436    2528 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13715 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3080 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


NameError: name 'quantile_loss' is not defined

In [26]:
# =========================
# EVALUATION (REQUIRED)
# =========================
july_mask = meta_month_val == 7

# --- Predict on validation set ---
y_val_pred = model.predict(X_val)
y_val_pred = y_val_pred.reshape(-1)

# --- Inverse scale anomalies ---
y_val_anom = y_scaler.inverse_transform(
    y_val.reshape(-1, 1)
).ravel()

y_val_pred_anom = y_scaler.inverse_transform(
    y_val_pred.reshape(-1, 1)
).ravel()

# --- Add monthly means back (physical units) ---
monthly_mean_dict = monthly_mean.to_dict()

y_val_phys = np.array([
    y_val_anom[i] + monthly_mean_dict[meta_month_val[i]]
    for i in range(len(y_val_anom))
])

y_val_pred_phys = np.array([
    y_val_pred_anom[i] + monthly_mean_dict[meta_month_val[i]]
    for i in range(len(y_val_pred_anom))
])

# --- Errors ---
errors = y_val_phys - y_val_pred_phys
abs_errors = np.abs(errors)

# =========================
# GLOBAL METRICS
# =========================
mae = abs_errors.mean()
rmse = np.sqrt(np.mean(errors ** 2))

print("\n================ EVALUATION =================")
print(f"Global MAE : {mae:.3f}")
print(f"Global RMSE: {rmse:.3f}")

# =========================
# MAE BY HORIZON
# =========================
print("\nMAE by Horizon:")
for h in sorted(np.unique(meta_horizon_val)):
    mask = meta_horizon_val == h
    h_mae = abs_errors[mask].mean()
    print(f"  h={h:2d} → MAE={h_mae:.3f}")

# =========================
# JULY-SPECIFIC DIAGNOSTICS
# =========================
july_mask = meta_month_val == 7

if july_mask.any():
    july_mae = abs_errors[july_mask].mean()
    print("\nJuly Diagnostics:")
    print(f"  July MAE: {july_mae:.3f}")

    # Peak ranking (Spearman-like via ranks)
    july_obs = y_val_phys[july_mask]
    july_pred = y_val_pred_phys[july_mask]

    if len(july_obs) > 2:
        obs_rank = np.argsort(np.argsort(july_obs))
        pred_rank = np.argsort(np.argsort(july_pred))
        rank_corr = np.corrcoef(obs_rank, pred_rank)[0, 1]
        print(f"  July Rank Correlation: {rank_corr:.3f}")
    else:
        print("  Not enough July samples for rank correlation")

else:
    print("\n(No July samples in validation set)")

# =========================
# OPTIONAL: EXTREME CAPTURE CHECK
# =========================
# Does the model underpredict the top 10%?
q90 = np.percentile(y_val_phys, 90)
extreme_mask = y_val_phys >= q90

if extreme_mask.any():
    extreme_bias = np.mean(y_val_pred_phys[extreme_mask] - y_val_phys[extreme_mask])
    print("\nExtreme-Event Diagnostics:")
    print(f"  Mean bias on top 10% events: {extreme_bias:.3f}")
else:
    print("\n(No extreme events in validation set)")


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step

================ EVALUATION =================
Global MAE : 0.097
Global RMSE: 0.120

MAE by Horizon:
  h= 1 → MAE=0.081
  h= 2 → MAE=0.113
  h= 3 → MAE=0.124
  h= 4 → MAE=0.074
  h= 5 → MAE=0.095
  h= 6 → MAE=0.110
  h= 7 → MAE=0.102
  h= 8 → MAE=0.105
  h= 9 → MAE=0.077
  h=10 → MAE=0.117
  h=11 → MAE=0.089
  h=12 → MAE=0.076

July Diagnostics:
  July MAE: 0.097
  July Rank Correlation: 0.089

Extreme-Event Diagnostics:
  Mean bias on top 10% events: -0.118
